# Exploración inicial del flujo de pacientes (tarea #13)

**Responsable:** Juan · Sprint 1

Primeras gráficas para ubicar **dónde parece haber congestión** y enfocar el Sprint 2.
Todavía no busca conclusiones.

**Antes de correr** (con la VPN, avisando en el grupo):
`python -m hun.data.extraer flujo_horario agregados conceptos`, luego `python -m hun.data.conceptos`.

Solo usa agregados de `data/raw/`. Las salidas se borran solas al hacer commit (`nbstripout`).
Las figuras se guardan en `reports/figures/exploracion/` y los hallazgos se pasan a `docs/hallazgos.md`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from hun import config
from hun.data.agregados import fecha_corte, serie_censo_etapas
from hun.data.conceptos import cargar

RAW = config.RAW_DIR
FIG = config.FIGURES_DIR / "exploracion"
FIG.mkdir(parents=True, exist_ok=True)
leer = lambda nombre: pd.read_parquet(RAW / f"{nombre}.parquet")  # noqa: E731
NOMBRE = cargar()
DIAS = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
PRINCIPALES = [581479, 9203, 3045729]


def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=150, bbox_inches="tight")


# Fecha de corte: después de ella solo hay registros sueltos (ver docs/inventario_base.md)
entradas = leer("entradas_por_hora")
CORTE = fecha_corte(entradas)
print("Fecha de corte:", CORTE.date())

## 1. Volumen por servicio, día de la semana y hora

In [ ]:
e = entradas.assign(hora=pd.to_datetime(entradas["hora"]))
e = e[(e["hora"] < CORTE) & e["visit_concept_id"].isin(PRINCIPALES)]
dias_totales = e["hora"].dt.normalize().nunique()

fig, axes = plt.subplots(1, len(PRINCIPALES), figsize=(16, 4), sharey=False)
for ax, tipo in zip(axes, PRINCIPALES, strict=True):
    sub = e[e["visit_concept_id"] == tipo]
    semanas = sub["hora"].dt.to_period("W").nunique()
    mapa = sub.groupby([sub["hora"].dt.day_of_week, sub["hora"].dt.hour])["entradas"].sum().unstack(
        fill_value=0
    ) / max(semanas, 1)
    im = ax.imshow(mapa.reindex(range(7)).fillna(0), aspect="auto", cmap="viridis")
    ax.set_yticks(range(7), DIAS)
    ax.set_xlabel("hora de llegada")
    ax.set_title(NOMBRE.get(tipo, tipo))
    fig.colorbar(im, ax=ax, label="llegadas promedio por semana")
fig.suptitle("¿Cuándo llegan los pacientes?")
plt.tight_layout()
guardar(fig, "01_llegadas_dia_hora")

In [ ]:
diario = (
    e.groupby([e["hora"].dt.normalize(), "visit_concept_id"])["entradas"]
    .sum()
    .unstack(fill_value=0)
)
fig, ax = plt.subplots(figsize=(14, 4))
diario.rolling(28).mean().rename(columns=lambda t: NOMBRE.get(t, t)).plot(ax=ax)
ax.set_ylabel("llegadas por día (media móvil 28 días)")
ax.set_title("Tendencia de la demanda")
guardar(fig, "02_tendencia")

## 2. ¿Quién llega? Sexo y grupo de edad

In [ ]:
perfil = leer("llegadas_diarias_perfil")
perfil = perfil[pd.to_datetime(perfil["fecha"]) < CORTE]
tabla = perfil.pivot_table(
    index=["visit_concept_id", "grupo_edad"],
    columns="gender_concept_id",
    values="visitas",
    aggfunc="sum",
)
pct = 100 * tabla / tabla.groupby(level=0).transform("sum").sum(axis=1).to_numpy()[:, None]
pct.rename(index=lambda t: NOMBRE.get(t, t), level=0).rename(
    columns=lambda g: NOMBRE.get(g, g)
).round(1)

## 3. Duración de las etapas según cuándo empiezan

Mediana (p50) de horas en cada etapa por día de la semana y hora de inicio. Las franjas más
claras son candidatas a cuello de botella.

In [ ]:
franja = leer("duracion_etapas_por_franja")
grupos = franja.groupby(["visit_detail_concept_id", "visit_concept_id"])["etapas"].sum()
grupos = grupos[grupos > 1000].sort_values(ascending=False).index[:4]
fig, axes = plt.subplots(1, len(grupos), figsize=(5 * len(grupos), 4), squeeze=False)
for ax, (etapa, visita) in zip(axes[0], grupos, strict=True):
    sub = franja[
        (franja["visit_detail_concept_id"] == etapa) & (franja["visit_concept_id"] == visita)
    ]
    mapa = sub.pivot_table(index="dia_semana", columns="hora", values="horas_p50")
    im = ax.imshow(mapa.reindex(range(1, 8)), aspect="auto", cmap="magma")
    ax.set_yticks(range(7), DIAS)
    ax.set_xlabel("hora de inicio de la etapa")
    ax.set_title(f"{NOMBRE.get(etapa, etapa)}\nen {NOMBRE.get(visita, visita)}")
    fig.colorbar(im, ax=ax, label="horas (p50)")
plt.tight_layout()
guardar(fig, "03_duracion_etapas_franja")

## 4. Esperas entre etapas

In [ ]:
trans = leer("transiciones_etapas")
trans = trans[pd.to_datetime(trans["mes"]) < CORTE]
# Promedio ponderado por pasos de las medianas mensuales (aproximación; el p50 exacto no se suma)
trans["peso"] = trans["pasos"]
resumen = (
    trans[trans["etapa_siguiente"] != -1]
    .groupby(["visit_concept_id", "etapa", "etapa_siguiente"])
    .apply(
        lambda g: pd.Series(
            {
                "pasos": g["pasos"].sum(),
                "pct_solapadas": 100 * g["solapadas"].sum() / g["pasos"].sum(),
                "espera_h_p50": (g["espera_h_p50"] * g["peso"]).sum() / g["peso"].sum(),
                "espera_h_p90": (g["espera_h_p90"] * g["peso"]).sum() / g["peso"].sum(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
for c in ["visit_concept_id", "etapa", "etapa_siguiente"]:
    resumen[c] = resumen[c].map(lambda t: NOMBRE.get(t, t))
resumen.sort_values("pasos", ascending=False).round(2)

## 5. Censo por etapa: pacientes adentro según la hora

El censo sale de `visit_detail` (en `visit_occurrence` el fin es igual al inicio).
Se muestra la mediana y el p90 del censo por hora del día: donde el p90 se separa mucho de
la mediana hay días de saturación.

In [ ]:
censo = serie_censo_etapas(leer("etapas_entradas_por_hora"), leer("etapas_salidas_por_hora"))
censo = censo[censo["ts"] < CORTE]
fig, axes = plt.subplots(1, censo["serie"].nunique(), figsize=(16, 3.5), squeeze=False)
for ax, (serie, sub) in zip(axes[0], censo.groupby("serie"), strict=True):
    por_hora = sub.groupby(sub["ts"].dt.hour)["y"]
    ax.plot(por_hora.median(), label="mediana")
    ax.plot(por_hora.quantile(0.9), label="p90", ls="--")
    ax.set_title(NOMBRE.get(serie, serie))
    ax.set_xlabel("hora del día")
axes[0][0].set_ylabel("pacientes adentro")
axes[0][-1].legend()
plt.tight_layout()
guardar(fig, "04_censo_etapas")

## 6. Dónde enfocar el Sprint 2

Anota en `docs/hallazgos.md` (solo agregados):
- Franjas de día y hora con más llegadas y con etapas más largas.
- Pasos entre etapas con mayor espera.
- Etapas y horas donde el p90 del censo se separa de la mediana.
- Diferencias por sexo o edad que valga la pena contrastar con la visita al hospital.